# Day 05: Plotting & Debugging — Reading Loss Curves

**Goal:** Learn to read loss curves so you can tell when training is healthy, broken, or overfitting.

You're about to start building real models (Day 06 onwards). When training goes wrong (and it WILL), the **loss curve** is your primary debugging tool.

Today we'll:
1. Build a proper training loop with train + validation tracking
2. Deliberately break training in 5 ways and see what each looks like
3. Learn the visual patterns so you can diagnose issues instantly

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

## 1. A Proper Training Loop — Tracking Train AND Validation Loss

In Day 2-3, we just plotted one loss curve. Real training tracks **two**:

- **Training loss** — how wrong on data the model SEES during training
- **Validation loss** — how wrong on data the model has NEVER seen

### Why both?

| Scenario | Train loss | Val loss | Meaning |
|----------|-----------|----------|---------|
| Both dropping | ↓ | ↓ | Healthy learning ✓ |
| Only train dropping | ↓ | ↑ | Memorizing (overfitting) |
| Neither dropping | = | = | Not learning (underfitting) |
| Train low, val low | ~0 | ~0 | Actually learned the pattern |

In [ ]:
# Generate data: y = 3x + 7 with noise
# We'll split into train (80%) and validation (20%)

X_all = torch.linspace(-5, 5, 100)
y_all = 3 * X_all + 7 + torch.randn(100) * 2

# Random shuffle and split
indices = torch.randperm(100)
train_idx = indices[:80]
val_idx = indices[80:]

X_train, y_train = X_all[train_idx], y_all[train_idx]
X_val, y_val = X_all[val_idx], y_all[val_idx]

print(f"Training samples:   {len(X_train)}")
print(f"Validation samples: {len(X_val)}")

plt.figure(figsize=(8, 4))
plt.scatter(X_train.numpy(), y_train.numpy(), alpha=0.6, label='Train', color='steelblue')
plt.scatter(X_val.numpy(), y_val.numpy(), alpha=0.8, label='Validation', color='tomato', marker='x', s=80)
plt.xlabel('x')
plt.ylabel('y')
plt.title('Our Data: Training vs Validation')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# A reusable training function that tracks BOTH losses

def train_model(lr=0.01, epochs=100, zero_grads=True, verbose=True):
    """Returns train_losses, val_losses, final weights"""
    w = torch.tensor(0.0, requires_grad=True)
    b = torch.tensor(0.0, requires_grad=True)
    optimizer = torch.optim.SGD([w, b], lr=lr)
    
    train_losses = []
    val_losses = []
    
    for epoch in range(epochs):
        # ---- TRAINING STEP ----
        y_pred_train = w * X_train + b
        train_loss = torch.mean((y_pred_train - y_train) ** 2)
        
        if zero_grads:
            optimizer.zero_grad()
        train_loss.backward()
        optimizer.step()
        
        # ---- VALIDATION STEP (no gradient updates!) ----
        with torch.no_grad():
            y_pred_val = w * X_val + b
            val_loss = torch.mean((y_pred_val - y_val) ** 2)
        
        train_losses.append(train_loss.item())
        val_losses.append(val_loss.item())
        
        if verbose and epoch % 20 == 0:
            print(f"Epoch {epoch:3d}: train_loss={train_loss.item():7.3f}, val_loss={val_loss.item():7.3f}")
    
    return train_losses, val_losses, (w.item(), b.item())

# Train with good settings
train_losses, val_losses, (w_final, b_final) = train_model(lr=0.01, epochs=100)
print(f"\nLearned: y = {w_final:.2f}x + {b_final:.2f}  (target: y = 3x + 7)")

### What a HEALTHY loss curve looks like:

In [ ]:
# Plot a healthy loss curve (with TWO curves: train and val)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Linear scale
ax1.plot(train_losses, 'b-', label='Training loss', linewidth=2)
ax1.plot(val_losses, 'r-', label='Validation loss', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('HEALTHY TRAINING (linear scale)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Log scale — reveals late-training behavior better
ax2.plot(train_losses, 'b-', label='Training loss', linewidth=2)
ax2.plot(val_losses, 'r-', label='Validation loss', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss (log scale)')
ax2.set_title('Same data, LOG SCALE reveals more')
ax2.set_yscale('log')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("What makes this healthy:")
print("  1. Both losses drop together (train and val)")
print("  2. Validation is slightly above training — normal and expected")
print("  3. Curves level off smoothly (converged)")
print("  4. Final loss values are small and close together\n")
print("PRO TIP: Always plot with log scale too. Linear hides late improvements.")

## 2. Training Gone Wrong — 4 Ways to Break It

Now let's DELIBERATELY break training in 4 different ways and see what each looks like. After this, you'll recognize these patterns instantly in your own work.

In [ ]:
# Train the same model under 4 different BROKEN settings
# and compare them all side by side

results = {}

# Baseline: healthy (for comparison)
print("Training 1/4: Healthy baseline (lr=0.01)...")
results['Healthy (lr=0.01)'] = train_model(lr=0.01, verbose=False)

# Problem 1: Learning rate too SMALL
print("Training 2/4: LR too small (lr=0.0001)...")
results['LR too small (0.0001)'] = train_model(lr=0.0001, verbose=False)

# Problem 2: Learning rate too LARGE
print("Training 3/4: LR too large (lr=0.5)...")
results['LR too large (0.5)'] = train_model(lr=0.5, verbose=False)

# Problem 3: Learning rate WAY too large → NaN
print("Training 4/4: LR way too large (lr=2.0) → NaN...")
results['LR way too large → NaN'] = train_model(lr=2.0, verbose=False)

print("\nAll 4 trainings complete. Let's plot them.")

In [ ]:
# Plot all 4 scenarios in a grid so we can compare patterns

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, (title, (train_l, val_l, weights)) in zip(axes, results.items()):
    # Replace NaN with a visible value so we can see the curve died
    train_l_plot = [v if not np.isnan(v) and not np.isinf(v) else 1e10 for v in train_l]
    val_l_plot = [v if not np.isnan(v) and not np.isinf(v) else 1e10 for v in val_l]
    
    ax.plot(train_l_plot, 'b-', label='Train', linewidth=2)
    ax.plot(val_l_plot, 'r-', label='Val', linewidth=2, alpha=0.7)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Annotate with final outcome
    final_train = train_l[-1]
    if np.isnan(final_train) or np.isinf(final_train):
        msg = "DEAD (NaN/Inf)"
        ax.set_ylim(0, 100)
    elif final_train > 50:
        msg = f"Didn't converge (loss={final_train:.1f})"
    else:
        msg = f"Converged (loss={final_train:.2f})"
    ax.text(0.05, 0.95, msg, transform=ax.transAxes, 
            verticalalignment='top', bbox=dict(boxstyle='round', alpha=0.3))

plt.tight_layout()
plt.show()

### Patterns to memorize:

**Healthy (top-left):** Smooth drop, levels off. Both lines close together. Low final loss. **This is the goal.**

**LR too small (top-right):** Curve barely moves — looks almost flat. Would need 10,000+ epochs to converge. **Fix: increase learning rate 10x-100x.**

**LR too large (bottom-left):** Curve goes DOWN then UP — loss oscillates or diverges. **Fix: decrease learning rate 10x.**

**LR way too large → NaN (bottom-right):** Loss explodes to infinity, becomes NaN within a few epochs. **Fix: decrease learning rate 100x-1000x.**

---

## 3. The Most Common Bug: Forgetting `zero_grad()`

In PyTorch, gradients **accumulate** by default. If you don't reset them, each epoch adds the new gradient to the old one, creating chaos.

Let's see what this looks like:

In [ ]:
# Train WITHOUT zero_grad() — the #1 beginner bug
print("Training without zero_grad() (the classic beginner bug):")
train_nz, val_nz, _ = train_model(lr=0.01, zero_grads=False, verbose=False)

# Compare
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(results['Healthy (lr=0.01)'][0], 'b-', label='WITH zero_grad() ✓', linewidth=2)

# Replace nan for plotting
train_nz_plot = [v if not np.isnan(v) and not np.isinf(v) else 1e10 for v in train_nz]
ax.plot(train_nz_plot, 'r-', label='WITHOUT zero_grad() ✗', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_yscale('symlog')  # handles both normal values and huge blowups
ax.set_title('Effect of missing zero_grad() — gradients accumulate, loss explodes')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

print("\nThe red curve looks like the 'LR too high → NaN' case!")
print("That's because accumulating gradients makes effective LR grow every epoch.")
print("\nIf your loss explodes, check:")
print("  1. Is optimizer.zero_grad() in your loop?")
print("  2. Is it BEFORE loss.backward()?")

## 4. Overfitting — When Train Loss Drops but Validation Goes Up

This is the most important pattern to recognize. It means the model is MEMORIZING the training data instead of LEARNING the underlying pattern.

Let's simulate overfitting deliberately:

### How to force overfitting:
- Use a small training set (model can memorize every point)
- Use a very complex model (more capacity = more memorization)
- Train for too many epochs

In [ ]:
# Simulate overfitting with a high-degree polynomial
# (instead of fitting a line, we fit a curvy polynomial that can memorize)

torch.manual_seed(1)
# Small training set so model can memorize
X_tiny = torch.linspace(-3, 3, 15)
y_tiny = 2 * X_tiny + 1 + torch.randn(15) * 1.5  # true: linear with noise

# Separate validation from a different random sample of the same distribution
X_valtiny = torch.linspace(-3, 3, 30)
y_valtiny = 2 * X_valtiny + 1 + torch.randn(30) * 1.5

# Fit a degree-10 polynomial (way too complex for linear data!)
degree = 10
# Use polynomial features: [1, x, x², x³, ... x^10]
def poly_features(x, degree):
    return torch.stack([x**i for i in range(degree + 1)], dim=1)

X_train_poly = poly_features(X_tiny, degree)      # shape (15, 11)
X_val_poly = poly_features(X_valtiny, degree)     # shape (30, 11)

# Model: just linear regression with polynomial features (11 weights)
weights = torch.zeros(degree + 1, requires_grad=True)
optimizer = torch.optim.SGD([weights], lr=0.0001)

train_losses_of = []
val_losses_of = []

for epoch in range(3000):
    # Training
    y_pred_train = X_train_poly @ weights
    train_loss = torch.mean((y_pred_train - y_tiny) ** 2)
    
    optimizer.zero_grad()
    train_loss.backward()
    optimizer.step()
    
    # Validation
    with torch.no_grad():
        y_pred_val = X_val_poly @ weights
        val_loss = torch.mean((y_pred_val - y_valtiny) ** 2)
    
    train_losses_of.append(train_loss.item())
    val_losses_of.append(val_loss.item())

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
ax1.plot(train_losses_of, 'b-', label='Training loss', linewidth=2)
ax1.plot(val_losses_of, 'r-', label='Validation loss', linewidth=2)
ax1.axvline(x=np.argmin(val_losses_of), color='green', linestyle='--', 
            label=f'Best val epoch = {np.argmin(val_losses_of)}')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('OVERFITTING: train loss keeps dropping, val loss starts rising!')
ax1.legend()
ax1.grid(True, alpha=0.3)

# The learned curve vs actual data
with torch.no_grad():
    x_smooth = torch.linspace(-3, 3, 200)
    y_smooth = poly_features(x_smooth, degree) @ weights

ax2.scatter(X_tiny.numpy(), y_tiny.numpy(), alpha=0.8, s=60, label='Training data', color='steelblue')
ax2.scatter(X_valtiny.numpy(), y_valtiny.numpy(), alpha=0.5, s=40, label='Validation data', color='tomato', marker='x')
ax2.plot(x_smooth.numpy(), y_smooth.numpy(), 'g-', linewidth=2, label='Overfit curve')
ax2.plot(x_smooth.numpy(), (2 * x_smooth + 1).numpy(), 'k--', linewidth=2, label='True pattern', alpha=0.5)
ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.set_title('The overfit model wiggles to hit every training point')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"At epoch 0:     train={train_losses_of[0]:.2f},  val={val_losses_of[0]:.2f}")
print(f"At best epoch:  train={train_losses_of[np.argmin(val_losses_of)]:.2f},  val={val_losses_of[np.argmin(val_losses_of)]:.2f}")
print(f"At final epoch: train={train_losses_of[-1]:.2f},  val={val_losses_of[-1]:.2f}")
print(f"\nTraining got 'better' but validation got WORSE — classic overfitting.")
print(f"Real pattern: y = 2x + 1 (a line)")
print(f"Model learned: a wiggly polynomial that goes through every training point exactly")

### Reading the overfitting plot:

**Left plot (loss curves):**
- Blue (training) keeps dropping — model is getting "better" on training data
- Red (validation) drops briefly, then STARTS RISING — model is getting WORSE on unseen data
- The green line marks the **best epoch** — that's where you should have STOPPED training

**Right plot (the model itself):**
- Blue dots = training data the model saw
- Red X's = validation data (never seen)
- Green curve = what the model learned — see how it twists through every training point?
- Dashed line = the true underlying pattern (a simple line)

**The model didn't learn the pattern — it memorized the noise.** When it encounters new data, it fails badly.

### How to prevent overfitting:

| Technique | What it does |
|-----------|-------------|
| **Early stopping** | Stop training when val loss starts rising |
| **More data** | Harder to memorize millions of examples |
| **Simpler model** | Less capacity = less memorization |
| **Regularization** | Penalize large weights (L1/L2) |
| **Dropout** | Randomly zero-out some neurons during training |

---

## 5. Running Average — Smoothing Noisy Loss Curves

When you train with small batches, loss is noisy (different batches give different losses). A running average makes trends visible:

In [ ]:
# Simulate noisy batch-level loss (common in real training)
torch.manual_seed(0)
np.random.seed(0)

# Fake noisy loss curve (decreasing trend with lots of noise)
n_steps = 500
true_trend = 5 * np.exp(-np.arange(n_steps) / 100) + 0.5  # decay curve
noise = np.random.randn(n_steps) * 0.8                      # random noise
noisy_loss = true_trend + noise

# Running average (smoothing)
def running_mean(x, window=50):
    x = np.array(x)
    return np.convolve(x, np.ones(window)/window, mode='valid')

smoothed_loss = running_mean(noisy_loss, window=30)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Raw (noisy)
ax1.plot(noisy_loss, alpha=0.5, linewidth=1, color='steelblue')
ax1.set_title('Raw loss per batch — NOISY, hard to see trend')
ax1.set_xlabel('Batch step')
ax1.set_ylabel('Loss')
ax1.grid(True, alpha=0.3)

# Smoothed
ax2.plot(noisy_loss, alpha=0.3, linewidth=1, color='steelblue', label='Raw')
ax2.plot(range(len(smoothed_loss)), smoothed_loss, linewidth=2, color='red', label='Smoothed (window=30)')
ax2.set_title('With running average — trend is CLEAR')
ax2.set_xlabel('Batch step')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("RULE OF THUMB: Always smooth your loss curves before analyzing.")
print("  - Per-batch losses: smooth with window ~50-200")
print("  - Per-epoch losses: usually no smoothing needed")

## 6. A Reusable TrainingTracker Class

Let's package all this into something we can reuse in all future days:

In [ ]:
# A utility class we can reuse in later days

class TrainingTracker:
    """Tracks train/val losses and produces diagnostic plots."""
    
    def __init__(self):
        self.train_losses = []
        self.val_losses = []
    
    def log(self, train_loss, val_loss=None):
        """Call this once per epoch (or batch)."""
        # Handle tensor or float
        if hasattr(train_loss, 'item'):
            train_loss = train_loss.item()
        if val_loss is not None and hasattr(val_loss, 'item'):
            val_loss = val_loss.item()
        self.train_losses.append(train_loss)
        if val_loss is not None:
            self.val_losses.append(val_loss)
    
    def plot(self, smooth_window=None, use_log=False):
        """Plot loss curves with optional smoothing and log scale."""
        fig, ax = plt.subplots(1, 1, figsize=(10, 5))
        
        def maybe_smooth(x):
            if smooth_window and len(x) > smooth_window:
                return np.convolve(x, np.ones(smooth_window)/smooth_window, mode='valid')
            return x
        
        train_plot = maybe_smooth(self.train_losses)
        ax.plot(train_plot, 'b-', label='Train', linewidth=2)
        
        if self.val_losses:
            val_plot = maybe_smooth(self.val_losses)
            ax.plot(val_plot, 'r-', label='Validation', linewidth=2)
        
        if use_log:
            ax.set_yscale('log')
        
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss')
        ax.set_title('Training Progress')
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.show()
    
    def diagnose(self):
        """Automatic diagnosis — prints what might be wrong."""
        t = self.train_losses
        v = self.val_losses if self.val_losses else None
        
        print("=" * 55)
        print("TRAINING DIAGNOSIS")
        print("=" * 55)
        
        if len(t) == 0:
            print("No data logged.")
            return
        
        # Check for NaN/Inf
        if any(np.isnan(x) or np.isinf(x) for x in t):
            print("❌ BROKEN: loss is NaN/Inf")
            print("   → Learning rate too high, or numerical blowup")
            print("   → Fix: decrease LR by 10-100x")
            return
        
        first = t[0]
        last = t[-1]
        best = min(t)
        
        # Check if loss decreased
        if last >= first * 0.9:
            print(f"⚠️  Loss barely moved: {first:.3f} → {last:.3f}")
            print("   → Either learning rate too small, or no gradients flowing")
            print("   → Fix: increase LR, or check .backward() is called")
        else:
            print(f"✓ Loss decreased: {first:.3f} → {last:.3f} (reduced by {(1-last/first)*100:.1f}%)")
        
        # Check for divergence
        if last > best * 2:
            print(f"⚠️  Loss is RISING — diverging")
            print(f"   → Best loss was {best:.3f}, now {last:.3f}")
            print(f"   → Fix: decrease LR")
        
        # Check for overfitting
        if v:
            final_gap = v[-1] - t[-1]
            if v[-1] > min(v) * 1.2:  # val loss rose 20% from best
                best_epoch = np.argmin(v)
                print(f"⚠️  OVERFITTING detected at epoch {len(v)-1}")
                print(f"   → Val loss best at epoch {best_epoch}: {v[best_epoch]:.3f}")
                print(f"   → Val loss now:  {v[-1]:.3f}")
                print(f"   → Fix: stop at epoch {best_epoch}, use more data, or simpler model")
            elif final_gap > t[-1]:
                print(f"⚠️  Large train/val gap: train={t[-1]:.3f}, val={v[-1]:.3f}")
                print(f"   → Possible mild overfitting — monitor closely")
            else:
                print(f"✓ Val loss tracks train loss — no overfitting detected")

# Demo: use it with our overfitting run
tracker = TrainingTracker()
for tl, vl in zip(train_losses_of, val_losses_of):
    tracker.log(tl, vl)

tracker.plot(use_log=True)
tracker.diagnose()

---

## Exercises

1. **Find the sweet spot LR:** Train with LRs 0.001, 0.003, 0.01, 0.03, 0.1, 0.3. Plot all 6 loss curves on one graph. Which LR converges fastest?

2. **Simulate gradient flow bug:** Set `requires_grad=False` on your weights. Train and plot the loss. What does it look like?

3. **Use the TrainingTracker:** Import the class, train any model, call `.diagnose()`. Does it correctly identify the issue?

4. **Early stopping:** Modify the training loop to STOP when val loss doesn't improve for 20 epochs. This is called "patience-based early stopping."

---

## Key Takeaways

### The 4 critical loss curve shapes

| Shape | Meaning | Fix |
|-------|---------|-----|
| Both drop smoothly, stay close | HEALTHY | Keep training |
| Flat/barely moving | LR too small OR no gradients | Increase LR; check backward() |
| Oscillating up and down | LR too large | Decrease LR 10x |
| Explodes to NaN/Inf | LR way too large OR numerical bug | Decrease LR 100x; check for exp/log |
| Train drops, val rises | OVERFITTING | Early stop, more data, regularize |

### The ML debugging workflow

```
Loss not decreasing?
  ├── Is loss NaN/Inf? → LR too high (decrease 10-100x)
  ├── Is loss constant? → Check optimizer.zero_grad(), loss.backward(), requires_grad
  └── Is loss slowly decreasing? → LR too small (increase 10x)

Loss decreasing but quality bad?
  ├── Train loss low, val loss high? → OVERFITTING
  ├── Both losses high? → UNDERFITTING (need bigger model)
  └── Losses fine, outputs wrong? → Check the forward pass logic
```

### The training loop you'll always write:

```python
for epoch in range(epochs):
    # Training
    y_pred = model(X_train)
    train_loss = loss_fn(y_pred, y_train)
    optimizer.zero_grad()
    train_loss.backward()
    optimizer.step()
    
    # Validation (no gradients!)
    with torch.no_grad():
        val_pred = model(X_val)
        val_loss = loss_fn(val_pred, y_val)
    
    # Track BOTH
    tracker.log(train_loss, val_loss)
```

---

**Tomorrow:** We start building neural networks! Day 06 — a single neuron learning logic gates (AND, OR, XOR).

From Day 6 onwards, we'll stop writing `w1`, `w2`, `b` explicitly. We'll use `nn.Linear` and start building multi-layer networks. The foundations are in place.